# Historical source sanity — 90 days

Execute in the persistent `fraud-ml-prod/historical-sanity` Kubeflow Notebook. Credentials come from `historical-source`; no credential values are displayed. The accepted dataset is the finite prod Argo Workflow output, not a locally generated substitute.

This source-level learnability experiment uses simulation truth for evaluation. It is not the governed, delayed-label training baseline in #53. Predictors contain checkout/attempt facts and strictly earlier history; no scenario, truth, risk decision or future outcome is a feature.


In [ ]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "duckdb==1.5.6",
        "pyarrow==24.0.0",
        "boto3==1.43.116",
        "scikit-learn==1.8.0",
        "pandas==3.0.6",
    ],
    check=True,
)

In [ ]:
import hashlib
import importlib.metadata
import json
import os
from pathlib import Path

import boto3
import duckdb
import matplotlib.pyplot as plt
import pandas as pd
import pyarrow.fs as fs
import pyarrow.parquet as pq
from botocore.exceptions import ClientError
from IPython.display import display
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

version = "v2"
endpoint = "minio.home.phuchoang.sbs"
source = fs.S3FileSystem(
    access_key=os.environ["AWS_ACCESS_KEY_ID"],
    secret_key=os.environ["AWS_SECRET_ACCESS_KEY"],
    region="us-east-1",
    scheme="https",
    endpoint_override=endpoint,
    force_virtual_addressing=False,
)
prefix = f"synthetic-source/historical/{version}"
with source.open_input_file(f"{prefix}/manifest.json") as stream:
    manifest_bytes = stream.read()
manifest = json.loads(manifest_bytes)
manifest_sha256 = hashlib.sha256(manifest_bytes).hexdigest()
assert manifest["dataset_version"] == version
cache = Path("source-cache") / version
cache.mkdir(parents=True, exist_ok=True)
fs.copy_files(
    prefix,
    str(cache.resolve()),
    source_filesystem=source,
    destination_filesystem=fs.LocalFileSystem(),
    use_threads=False,
)
report = json.loads((cache / manifest["validation_report"]).read_text())
assert all(check["result"] in ("passed", "info") for check in report["checks"])
provenance = {
    "dataset_version": version,
    "manifest_sha256": manifest_sha256,
    "seed": manifest["seed"],
    "generator_revision": manifest["generator_revision"],
    "contract": manifest["contract"],
    "history_end": manifest["history_end"],
    "native_checks": len(report["checks"]),
    "versions": {
        name: importlib.metadata.version(name)
        for name in ("duckdb", "pyarrow", "pandas", "scikit-learn", "matplotlib")
    },
}
display(provenance)

## Storage, quality and physical cutover

Count physical rows before deduplication, compare each date partition with the manifest, inspect nested Arrow layouts, and measure duplicate content independently. Labels are partitioned by availability time. Complete histories span event partitions.


In [ ]:
con = duckdb.connect(config={"threads": 2, "memory_limit": "1GiB"})
for name in ("payment_events", "fraud_labels"):
    paths = sorted((cache / name).rglob("*.parquet"))
    con.read_parquet([str(p) for p in paths], union_by_name=True, hive_partitioning=False).create_view(name)
    partition_name = "occurred_date" if name == "payment_events" else "label_available_date"
    actual = {}
    for partition in sorted((cache / name).glob(f"{partition_name}=*")):
        actual[partition.name.split("=", 1)[1]] = sum(
            pq.ParquetFile(p).metadata.num_rows for p in partition.glob("*.parquet")
        )
    assert actual == manifest["partitions"][name]
cutover = manifest["configured"]["schema_cutover"]
physical = []
for path in sorted((cache / "payment_events").rglob("*.parquet")):
    file = pq.ParquetFile(path)
    day = path.parent.name.split("=", 1)[1]
    signals = file.schema_arrow.field("attempt_started").type.field("signals").type
    present = "user_agent_family" in [field.name for field in signals]
    assert present == (day >= cutover)
    physical.append(
        {
            "date": day,
            "user_agent_column": present,
            "rows": file.metadata.num_rows,
            "row_groups": file.metadata.num_row_groups,
            "codec": file.metadata.row_group(0).column(0).compression,
        }
    )
duplicates = con.sql("SELECT count(*) - count(DISTINCT event_id) FROM payment_events").fetchone()[0]
row_count = con.sql("SELECT count(*) FROM payment_events").fetchone()[0]
assert duplicates == manifest["observed"]["duplicate_events"]
conflicts = con.sql(
    "SELECT count(*) FROM (SELECT event_id FROM payment_events e GROUP BY event_id HAVING count(DISTINCT to_json(e)) > 1)"
).fetchone()[0]
assert conflicts == 0
cutover_nulls = con.sql("""SELECT CAST(occurred_at AS DATE) AS day,
  count(*) AS attempts,
  count(attempt_started.signals.user_agent_family) AS user_agent_values
FROM payment_events WHERE attempt_started IS NOT NULL GROUP BY day ORDER BY day""").df()
assert all(
    (row.user_agent_values == 0 if str(row.day.date()) < cutover else row.user_agent_values == row.attempts)
    for row in cutover_nulls.itertuples()
)
display(cutover_nulls)
display(pd.DataFrame(physical))
display(
    {
        "physical_events": row_count,
        "duplicate_rows": duplicates,
        "duplicate_rate": duplicates / row_count,
        "storage": manifest["storage"],
    }
)

In [ ]:
con.sql("CREATE VIEW canonical AS SELECT DISTINCT * FROM payment_events")
con.sql("""CREATE VIEW attempts AS
SELECT e.integration_id,e.payment_id,e.attempt_started.attempt_id AS attempt_id,
  e.occurred_at AS attempted_at,
  c.payment_created.customer.customer_id AS customer_id,
  c.payment_created.merchant.merchant_id AS merchant_id,
  c.payment_created.snapshot.total.minor_units AS amount,
  c.payment_created.snapshot.merchant_category AS category,
  date_diff('day',c.payment_created.snapshot.customer_account_created_at,e.occurred_at) AS account_age_days,
  CAST(e.attempt_started.signals.ip_country <> c.payment_created.snapshot.shipping.country AS INTEGER) AS foreign_ip,
  CAST(c.payment_created.snapshot.shipping.postal_code = '33166' AS INTEGER) AS forwarder,
  e.attempt_started.signals.device_fingerprint AS device,
  e.attempt_started.signals.ip_network_fingerprint AS network,
  e.attempt_started.signals.payment_method.fingerprint AS method,
  c.payment_created.snapshot.shipping.address_fingerprint AS address,
  t.is_fraud,t.scenario
FROM canonical e JOIN canonical c USING(integration_id,payment_id)
JOIN fraud_labels t ON t.integration_id=e.integration_id AND t.payment_id=e.payment_id
  AND t.attempt_id=e.attempt_started.attempt_id AND t.source='LABEL_SOURCE_SIMULATION_TRUTH'
WHERE e.attempt_started IS NOT NULL AND c.payment_created IS NOT NULL""")
summary = con.sql(
    "SELECT scenario,count(*) AS attempts,avg(CAST(is_fraud AS DOUBLE)) AS fraud_rate FROM attempts GROUP BY scenario ORDER BY attempts DESC"
).df()
distincts = con.sql(
    "SELECT approx_count_distinct(device) devices,approx_count_distinct(network) networks,approx_count_distinct(address) addresses,approx_count_distinct(method) methods FROM attempts"
).df()
merchant_skew = con.sql(
    "SELECT merchant_id,count(*) AS attempts FROM attempts GROUP BY merchant_id ORDER BY attempts DESC"
).df()
label_delays = con.sql(
    "SELECT source,count(*) AS labels,min(date_diff('day',occurred_at,label_available_at)) AS min_delay_days,max(date_diff('day',occurred_at,label_available_at)) AS max_delay_days FROM fraud_labels GROUP BY source"
).df()
display(summary)
display(distincts)
display(label_delays)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
con.sql("SELECT amount FROM attempts").df()["amount"].plot.hist(
    bins=60, logy=True, ax=axes[0], title="Amount in minor units"
)
con.sql("SELECT category,count(*) AS attempts FROM attempts GROUP BY category ORDER BY attempts DESC").df().plot.bar(
    x="category", y="attempts", ax=axes[1], legend=False, title="Category skew"
)
merchant_skew.head(15).plot.bar(x="merchant_id", y="attempts", ax=axes[2], legend=False, title="Top merchant skew")
axes[2].set_xticklabels([])
plt.tight_layout()
plt.show()

## Ninety-day drift, history and descriptor provenance

Read the completed production output, compare daily unique purchase counts with the manifest, and compare baseline and post-ramp rates. Verify stable entity creation timestamps, an independent thirty-day lookback and delayed-label availability at history end. Check both descriptor hashes against the versioned files shipped with this notebook.


In [ ]:
from datetime import date, datetime, timedelta

expected = Path('history90-descriptors')
for field, filename in [('contract','historical-payments.odcs.yaml'),('data_product','historical-payments.odps.yaml')]:
    assert hashlib.sha256((expected / filename).read_bytes()).hexdigest() == manifest[field]['sha256']
assert hashlib.sha256((expected / 'historical-90d.yaml').read_bytes()).hexdigest() == manifest['config_sha256']
assert manifest['generator_revision'] == '3f18bf10d710d4bf65d23e5f6e86d9926984e712e0dcdb54c20961d472cb07a0'
provenance['data_product'] = manifest['data_product']
con.sql("""CREATE VIEW purchases AS SELECT DISTINCT integration_id,payment_id,occurred_at,
 payment_created.customer.customer_id AS customer_id,
 payment_created.merchant.merchant_id AS merchant_id,
 payment_created.snapshot.customer_account_created_at AS customer_created_at,
 payment_created.snapshot.merchant_account_created_at AS merchant_created_at
 FROM payment_events WHERE payment_created IS NOT NULL""")
counts = dict(con.sql('SELECT CAST(occurred_at AS DATE)::VARCHAR,count(*) FROM purchases GROUP BY 1 ORDER BY 1').fetchall())
configuration = manifest['configuration']
start = date.fromisoformat(configuration['start_date'])
daily = {(start + timedelta(days=i)).isoformat(): counts.get((start + timedelta(days=i)).isoformat(),0)
         for i in range(configuration['duration_days'])}
assert daily == manifest['observed']['daily_purchase_counts'] and len(daily) == 90
customers = configuration['counts']['customers']
drift = configuration['drift']
drift_start = date.fromisoformat(drift['start_date'])
ramp_end = drift_start + timedelta(days=drift['ramp_days'])
baseline = [count for day,count in daily.items() if date.fromisoformat(day) < drift_start]
post = [count for day,count in daily.items() if date.fromisoformat(day) >= ramp_end]
ratio = (sum(post)/len(post))/(sum(baseline)/len(baseline))
assert 1.4 < ratio < 1.6
for identity,column in [('customer_id','customer_created_at'),('merchant_id','merchant_created_at')]:
    assert con.sql(f'SELECT count(*) FROM (SELECT {identity} FROM purchases GROUP BY integration_id,{identity} HAVING count(DISTINCT {column})<>1)').fetchone()[0] == 0
sample = con.execute("""SELECT integration_id,customer_id,occurred_at,prior_count FROM (
 SELECT *,count(*) OVER (PARTITION BY integration_id,customer_id ORDER BY occurred_at
 RANGE BETWEEN INTERVAL 30 DAYS PRECEDING AND INTERVAL 1 MICROSECOND PRECEDING) AS prior_count
 FROM purchases) WHERE occurred_at>=? ORDER BY occurred_at,payment_id LIMIT 1""",
 [datetime.combine(start + timedelta(days=60),datetime.min.time())]).fetchone()
integration,customer,at,prior = sample
independent = con.execute('SELECT count(*) FROM purchases WHERE integration_id=? AND customer_id=? AND occurred_at>=? AND occurred_at<?',
 [integration,customer,at-timedelta(days=30),at]).fetchone()[0]
assert prior == independent and prior > 0
cutoff = datetime.fromisoformat(manifest['history_end'])
chargebacks,available,future = con.execute("""SELECT count(*),count(*) FILTER (WHERE label_available_at<=?),
 count(*) FILTER (WHERE label_available_at>?) FROM fraud_labels WHERE source='LABEL_SOURCE_CHARGEBACK'""",[cutoff,cutoff]).fetchone()
assert available > 0 and future > 0 and available+future == chargebacks
history_extension = {'purchase_days':len(daily),'purchases':sum(daily.values()),
 'baseline':{'days':len(baseline),'purchases_per_customer_day':sum(baseline)/len(baseline)/customers},
 'post_ramp':{'days':len(post),'purchases_per_customer_day':sum(post)/len(post)/customers},
 'observed_frequency_ratio':ratio,'stable_customer_and_merchant_creation_times':True,
 'thirty_day_lookup':{'decision_time':at.isoformat(),'window_start':(at-timedelta(days=30)).isoformat(),
                      'prior_purchases':prior,'independent_query_matches':True},
 'label_cutoff':{'cutoff':cutoff.isoformat(),'chargebacks':chargebacks,'available':available,'excluded_future':future},
 'odcs_odps_config_hashes_match':True,'generator_revision_matches':True}
display(history_extension)
fig,ax = plt.subplots(figsize=(12,4))
pd.Series(daily).plot(ax=ax,title='Daily unique purchases across ninety simulated dates')
ax.axvline(drift_start.isoformat(),color='orange',label='Drift starts')
ax.axvline(ramp_end.isoformat(),color='red',label='Ramp complete')
ax.set_ylabel('Unique purchases')
ax.legend()
plt.tight_layout()
plt.show()

## Learnability from facts versus earlier history

Train on the dates 31–60 after a thirty-day warmup and evaluate on dates 61–90. Exclude the first thirty dates from model fitting and evaluation while retaining them as prior history. Compute history using only timestamps strictly before each attempt. Both models have the same settings and exclude raw identifiers. Simulation truth is used solely as the target; this diagnostic does not claim production model accuracy or delayed-label availability at the split.


In [ ]:
frame = con.sql("""SELECT *,
  count(*) OVER (PARTITION BY customer_id ORDER BY attempted_at RANGE BETWEEN INTERVAL 30 DAYS PRECEDING AND INTERVAL 1 MICROSECOND PRECEDING) AS customer_prior_attempts,
  avg(amount) OVER (PARTITION BY customer_id ORDER BY attempted_at RANGE BETWEEN INTERVAL 30 DAYS PRECEDING AND INTERVAL 1 MICROSECOND PRECEDING) AS customer_prior_amount,
  count(DISTINCT customer_id) OVER (PARTITION BY device ORDER BY attempted_at RANGE BETWEEN INTERVAL 1 DAY PRECEDING AND INTERVAL 1 MICROSECOND PRECEDING) AS device_prior_customers,
  count(DISTINCT method) OVER (PARTITION BY device ORDER BY attempted_at RANGE BETWEEN INTERVAL 1 HOUR PRECEDING AND INTERVAL 1 MICROSECOND PRECEDING) AS device_prior_methods,
  count(DISTINCT customer_id) OVER (PARTITION BY address ORDER BY attempted_at RANGE BETWEEN INTERVAL 1 DAY PRECEDING AND INTERVAL 1 MICROSECOND PRECEDING) AS address_prior_customers,
  count(*) OVER (PARTITION BY payment_id ORDER BY attempted_at RANGE BETWEEN UNBOUNDED PRECEDING AND INTERVAL 1 MICROSECOND PRECEDING) AS payment_prior_attempts,
  count(*) OVER (PARTITION BY customer_id,device ORDER BY attempted_at RANGE BETWEEN UNBOUNDED PRECEDING AND INTERVAL 1 MICROSECOND PRECEDING) AS customer_device_prior_attempts
FROM attempts ORDER BY attempted_at,payment_id,attempt_id""").df()
frame["amount_over_prior"] = frame["amount"] / frame["customer_prior_amount"].replace(0, float("nan"))
category_columns = pd.get_dummies(frame["category"], prefix="category", dtype=int)
frame = frame.join(category_columns)
single = ["amount", "account_age_days", "foreign_ip", "forwarder", *category_columns.columns]
history = single + [
    "customer_prior_attempts",
    "device_prior_customers",
    "device_prior_methods",
    "address_prior_customers",
    "payment_prior_attempts",
    "customer_device_prior_attempts",
    "amount_over_prior",
]
split = pd.Timestamp(manifest["configuration"]["start_date"], tz="UTC") + pd.Timedelta(days=60)
train = frame["attempted_at"] < split
target = frame["is_fraud"].astype(int)
assert target[train].nunique() == target[~train].nunique() == 2
metrics = []
for name, columns in [("single_attempt", single), ("with_history", history)]:
    model = HistGradientBoostingClassifier(
        max_iter=100, max_leaf_nodes=15, min_samples_leaf=30, early_stopping=False, random_state=manifest["seed"]
    )
    model.fit(frame.loc[train, columns], target[train])
    scores = model.predict_proba(frame.loc[~train, columns])[:, 1]
    metrics.append(
        {
            "features": name,
            "average_precision": average_precision_score(target[~train], scores),
            "roc_auc": roc_auc_score(target[~train], scores),
        }
    )
metrics = pd.DataFrame(metrics).set_index("features")
display(
    {
        "train_attempts": int(train.sum()),
        "test_attempts": int((~train).sum()),
        "test_fraud_rate": float(target[~train].mean()),
        "split": str(split),
    }
)
display(metrics)
improvement = float(
    metrics.loc["with_history", "average_precision"] - metrics.loc["single_attempt", "average_precision"]
)
learnability = {
    "average_precision_gain": improvement,
    "single_attempt_not_near_perfect": bool(metrics.loc["single_attempt", "average_precision"] < 0.95),
    "history_improves": improvement > 0.02,
}
display(learnability)
assert learnability["single_attempt_not_near_perfect"] and learnability["history_improves"]

## Shared source access and retained evidence

The workflow's successful publication proves scoped writes, and the Notebook read proves source consumption. Exercise denied deletion against a never-created key, denied writes outside the allowed prefix, and denied access to the pipeline artifact bucket. No accepted dataset object is modified.


In [ ]:
s3 = boto3.client(
    "s3",
    endpoint_url=f"https://{endpoint}",
    region_name="us-east-1",
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
)
denials = {}
for name, operation in {
    "delete_source": lambda: s3.delete_object(
        Bucket="synthetic-source", Key="historical/_verification/never-created-v3-history90"
    ),
    "write_outside_prefix": lambda: s3.put_object(
        Bucket="synthetic-source", Key="_verification-outside-prefix-v3-history90", Body=b"permission probe"
    ),
    "read_pipeline_bucket": lambda: s3.list_objects_v2(Bucket="kubeflow-prod-pipelines", MaxKeys=1),
}.items():
    try:
        operation()
    except ClientError as error:
        denials[name] = error.response["Error"]["Code"]
    else:
        raise AssertionError(f"{name} unexpectedly permitted")
assert all(value == "AccessDenied" for value in denials.values())
display(denials)
evidence = {
    **provenance,
    "learnability": learnability,
    "metrics": metrics.reset_index().to_dict(orient="records"),
    "permissions": denials,
    "observed": manifest["observed"],
}
Path("historical-90d-sanity-evidence.json").write_text(json.dumps(evidence, indent=2))
print("Sanity checks passed; evidence retained on the Notebook PVC.")